# Step 2.6: the Rust engine on ticks against the reference engine

This notebook is the check of step 2.6. The Rust engine now fills market, stop-market, and limit orders on trade prints and TBBO records, and it must fill them exactly as the Python reference engine does, which implements `docs/spec/fill-semantics.md` rule by rule. Until the Python bindings of step 2.10, the notebook reaches the Rust engine through its command-line runner, `fefat-cli run`, which `just build-engine` builds.

It runs the scenarios of `fixtures/scenarios/` on ticks through both engines, shows S-fill-012, the gap through a stop, and then runs tapes that Hypothesis draws through both, counting the differences. Nothing here reads your store: the tapes are the scenarios' and the drawn ones.

## The two engines

The Rust engine is the binary `just build-engine` writes into the repository's `engine/target/release/`; set `FEFAT_CLI` to use another. The reference engine is `fefat.reference.engine`.

Expected on this PC: `The Rust engine: ...\engine\target\release\fefat-cli.exe`.

In [ ]:
import os
from pathlib import Path

from hypothesis import given, seed, settings
from hypothesis import strategies as st

import fefat
from fefat.config import resolve_paths
from fefat.market.fidelity import TickFidelity
from fefat.reference.generated_cases import drawn_cases
from fefat.reference.rust_run import (
    RustEngine,
    compare_engines,
    default_binary,
    engine_cases,
    explained,
)
from fefat.reference.scenarios import load_scenarios

REPOSITORY = Path(fefat.__file__).resolve().parents[4]
engine = RustEngine(default_binary())
print(f"The Rust engine: {engine.binary}")

## The scenarios on ticks

Every scenario whose tape is trades or TBBO records, 23 of them, runs through both engines: the Rust engine's as one batch, in one process. For each, the two engines must give the same fills, price, time, rule, flags, liquidity, slippage, and cash, and the same event log, entry by entry. The 24 scenarios on bars wait for step 2.7, which replays bars.

Expected: `23 scenarios on ticks: 23 agree with the reference engine`.

In [ ]:
scenarios = load_scenarios(REPOSITORY / "fixtures" / "scenarios")
on_ticks = [scenario for scenario in scenarios if isinstance(scenario.fidelity, TickFidelity)]
cases = engine_cases(on_ticks, resolve_paths(os.environ).cache_dir)
comparisons = {
    comparison.case.scenario.id: comparison for comparison in compare_engines(engine, cases)
}
agreeing = [
    scenario_id for scenario_id, comparison in comparisons.items() if not comparison.differences()
]
print(f"{len(on_ticks)} scenarios on ticks: {len(agreeing)} agree with the reference engine")
for scenario_id, comparison in comparisons.items():
    for difference in comparison.differences():
        print(f"{scenario_id}: {difference}")

## S-fill-012, a gap through a stop

A long's sell stop at 4995.00 sees trades at 4996.00 and 4995.50, then 4990.25: the market jumped past the stop without trading at it. F-STP-03 fills a stop on trades at the worse of its price moved by the slippage tick, 4994.75, and the trade that triggered it, so at the gap price, 4990.25, with no tick added beyond it.

Expected: the tape and the order as the scenario writes them, the expected fill `order 1 at 4990.25 at 2026-03-06T14:30:01.25Z (F-STP-03)`, and the same fill from both engines, 19 ticks past the stop.

In [ ]:
print(explained(comparisons["S-fill-012"]))

## Drawn tapes

Hypothesis draws tapes the scenarios do not hold: random walks of trades or TBBO records with jumps as wide as a session's range, records at one instant, tapes past midnight, breaks of the market, and every setting. Half the cases place one to four orders with OCO pairs and brackets; the other half run the execution template's basic subset over drawn signals, with its brackets in ticks, as R multiples, or at the signal frame's prices, and its signal exits. The test suite runs 10,000 of them; this cell runs 1,000 with the same seed, ten to a batch, and counts the tapes on which the engines differ in their fills, their event logs, or the position and cash they end with.

Expected: `1000 drawn tapes through both engines`, then `0 differences`, in about five seconds.

In [ ]:
DIFFERENTIAL_SEED = 20_261_026
drawn, differing = [], []


@seed(DIFFERENTIAL_SEED)
@settings(max_examples=100, deadline=None, database=None)
@given(cases=st.lists(drawn_cases(), min_size=10, max_size=10))
def compare_drawn_tapes(cases):
    comparisons = compare_engines(engine, cases)
    drawn.extend(comparisons)
    differing.extend(comparison for comparison in comparisons if comparison.differences())


compare_drawn_tapes()
for comparison in differing[:3]:
    print(explained(comparison), *comparison.differences(), sep="\n")
print(f"{len(drawn)} drawn tapes through both engines")
print(f"{len(differing)} differences")

## Your checks

1. Open `fixtures/scenarios/S-fill-012.yaml`: the stop is at 4995.00 and the next trade past it prints 4990.25, so the fill must be 4990.25. The cell on S-fill-012 shows both engines filling it there.
2. The last cell prints `0 differences` for the drawn tapes.